# Лабораторная работа №11: Alignment VLM по предпочтениям

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Собрать preference pairs без length/style shortcut.
- Провести LoRA preference optimization на 2B fallback.
- Сравнить base/SFT/aligned по win rate и hallucinations.


## Теоретическая часть

Preference optimization использует пары chosen/rejected. DPO увеличивает относительную вероятность chosen относительно reference policy без отдельной reward model. Для VLM важно, чтобы изображения и prompts совпадали в обеих ветвях.


### Математическая постановка

$$\mathcal L_{DPO}=-\log\sigma\left(\beta[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}]\right).$$


### Материалы

- [DPO paper](https://arxiv.org/abs/2305.18290)
- [TRL цикл оптимизации DPO](https://huggingface.co/docs/trl/dpo_trainer)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Preference dataset является экспертным входом; пример с несуществующим cats.jpg не используется. Нужен адаптер lab 9 и GPU. При смене версии TRL сверьте мультимодальный цикл оптимизации DPO и зафиксируйте версии. `FAST_DEV_RUN=True` выполняет реальные два шага DPO, но не является evidence качества. Без заполненного слепого экспертного аудита итоговая метрика не создаётся.

In [1]:
# --- работаем полностью офлайн: модель и датасет берутся из локального кэша ---
# Если полный manifest.jsonl ещё не скачан: временно поставьте "0" в флагах ниже,
# выполните ЭТАП 0 (скачивание ScienceQA) и верните "1".
import os
os.environ["HF_HUB_OFFLINE"] = "1"        # huggingface_hub: в сеть не ходить, брать из кэша
os.environ["TRANSFORMERS_OFFLINE"] = "1"  # transformers: то же самое
os.environ["HF_DATASETS_OFFLINE"] = "1"   # datasets: то же самое

# utf-8 shim: запись/чтение файлов по умолчанию в utf-8 (запускать первой после Restart Kernel)
import pathlib

def _install_utf8_shim():
    if getattr(pathlib.Path.write_text, "_utf8_shim", False):
        return
    _wt, _rt = pathlib.Path.write_text, pathlib.Path.read_text
    def write_text(self, data, *args, **kwargs):
        kwargs.setdefault("encoding", "utf-8")
        return _wt(self, data, *args, **kwargs)
    def read_text(self, *args, **kwargs):
        kwargs.setdefault("encoding", "utf-8")
        return _rt(self, *args, **kwargs)
    write_text._utf8_shim = read_text._utf8_shim = True
    pathlib.Path.write_text, pathlib.Path.read_text = write_text, read_text

_install_utf8_shim()
print("utf-8 shim установлен")

utf-8 shim установлен


In [2]:
from pathlib import Path
import json, random
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))
import hashlib
from datasets import load_dataset

SCIENCEQA_ID = 'derek-thomas/ScienceQA'
SCIENCEQA_REVISION = 'f18b0a70359ebfb41f658fd564208d0355b013f4'

def prepare_scienceqa(directory, smoke=False):
    """Fixed official splits, image-level deduplication, explicitly defined shifts."""
    directory = Path(directory).resolve(); directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / ('manifest-smoke.jsonl' if smoke else 'manifest.jsonl')
    limits = {'train': 8 if smoke else 512, 'validation': 4 if smoke else 64,
              'id_test': 4 if smoke else 64, 'ood_test': 4 if smoke else 64,
              'general_test': 4 if smoke else 64}
    if manifest.exists():
        records = [json.loads(s) for s in manifest.read_text().splitlines()]
        if all(Path(r['image']).is_file() for r in records) and all(sum(r['split']==s for r in records)==n for s,n in limits.items()):
            return manifest
    records, seen, counts = [], set(), dict.fromkeys(limits, 0)
    for official in ('train', 'validation', 'test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= limits[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = directory / f'{digest}.png'; image.save(path)
            answer = int(row['answer']); choices = row['choices']
            question = row['question']+'\n'+'\n'.join(f'{chr(65+i)}. {v}' for i,v in enumerate(choices))+'\nReturn only the letter of the correct option.'
            records.append({'id': f'scienceqa-{official}-{index}', 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+answer),
                            'choices': choices, 'answer_index': answer, 'label_type': 'multiple_choice',
                            'license': 'MIT (code license, lupantech/ScienceQA)',
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject':row['subject'], 'grade':row['grade'], 'image_sha256':digest})
            counts[split] += 1
            required = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == limits[s] for s in required): break
    if counts != limits: raise RuntimeError(f'Insufficient licensed image examples: {counts}; expected {limits}')
    manifest.write_text('\n'.join(json.dumps(r,ensure_ascii=False) for r in records)+'\n')
    (directory/'ATTRIBUTION.md').write_text('ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: '+SCIENCEQA_ID+'\nRevision: '+SCIENCEQA_REVISION+'\nDataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/\nOriginal project: https://scienceqa.github.io/\nImage subsets and manifests retain attribution; downloaded images are not committed to this course.\nID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.\n')
    return manifest


In [3]:
# ЭТАП 0 — окружение, пути, данные
import importlib.metadata as _imd, platform, torch
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
PFX = "_fast" if FAST_DEV_RUN else ""
DATA_DIR = Path("../lab9/data/lab9")            # те же данные, что в лабах 9/10
if not (DATA_DIR / "manifest.jsonl").is_file():
    DATA_DIR = Path("data/lab9")

manifest = prepare_scienceqa(DATA_DIR, smoke=FAST_DEV_RUN)
records = [json.loads(s) for s in manifest.read_text(encoding="utf-8").splitlines() if s.strip()]
_limits = {"train": 8 if FAST_DEV_RUN else 512, "validation": 4 if FAST_DEV_RUN else 64,
           "id_test": 4 if FAST_DEV_RUN else 64, "ood_test": 4 if FAST_DEV_RUN else 64,
           "general_test": 4 if FAST_DEV_RUN else 64}
counts = {}
for r in records: counts[r["split"]] = counts.get(r["split"], 0) + 1
assert all(sum(r["split"] == s for r in records) == n for s, n in _limits.items()), counts

versions = {p: _imd.version(p) for p in
            ("unsloth", "unsloth_zoo", "transformers", "trl", "peft", "torch", "bitsandbytes")}
save_json(f"envs11{PFX}.json", {
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if torch.cuda.is_available() else 0,
    "os": platform.platform(), "model": BASE_MODEL + " (4-bit NF4, локальный кэш)",
    "fast_dev_run": FAST_DEV_RUN, "data_manifest": str(manifest),
    "dpo": {"beta": 0.1, "lr": 5e-6, "accum": 8, "optimizer_steps": 2 if FAST_DEV_RUN else 40, "micro_batch": 1, "seed": SEED},
    "versions": versions,
})
print("манифест:", manifest)
print("записей:", len(records), "| по сплитам:", counts)
print("версии:", versions)
print("ЭТАП 0 OK:", "FAST-режим, артефакты с суффиксом _fast" if PFX else "полный режим")

манифест: C:\Users\eric1\Desktop\SOTA\lab9\data\lab9\manifest.jsonl
записей: 768 | по сплитам: {'train': 512, 'validation': 64, 'ood_test': 64, 'general_test': 64, 'id_test': 64}
версии: {'unsloth': '2026.10.8', 'unsloth_zoo': '2026.10.8', 'transformers': '4.57.6', 'trl': '0.29.1', 'peft': '0.21.2', 'torch': '2.14.1+cu126', 'bitsandbytes': '0.50.2'}
ЭТАП 0 OK: полный режим


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Предпочтения из визуальных ответов

*Вес: 1 балл.*

**Постановка.** Подготовьте ScienceQA через открытый загрузчик. Для каждого изображения chosen — правильная буква, rejected — следующий неверный вариант. Явно пометьте: это предпочтения на основе эталонной разметки, а не экспертные оценки полезности.

**Результат.** Список пар с provenance.

**Критерий принятия.** Изображения существуют, ответы различаются и соответствуют вариантам; лицензия и источник сохранены.


In [4]:
# ЭТАП 1 — preference-пары из эталонной разметки (НЕ экспертные оценки полезности)
# chosen = правильная буква, rejected = следующая по кругу НЕВЕРНАЯ буква из choices
pairs_all = []
for r in records:
    ch = list(r["choices"]); n = len(ch); ai = int(r["answer_index"])
    assert 0 <= ai < n and r["answer"] == chr(65 + ai)
    rj = (ai + 1) % n                                  # «следующий неверный вариант»
    assert rj != ai and chr(65 + rj) != r["answer"]
    assert Path(r["image"]).is_file(), f"нет картинки: {r['image']}"
    pairs_all.append({
        "id": r["id"], "split": r["split"], "image": r["image"], "image_sha256": r["image_sha256"],
        "question": r["question"], "choices": ch, "answer_index": ai,
        "chosen": r["answer"], "rejected": chr(65 + rj), "rejected_offset": (rj - ai) % n,
        "provenance": {
            "dataset": SCIENCEQA_ID, "revision": SCIENCEQA_REVISION,
            "license": r["license"], "source_url": r["source_url"],
            "label_type": "label-derived preference (эталонная разметка, НЕ экспертная оценка полезности)",
        },
    })

assert all(p["chosen"] != p["rejected"] for p in pairs_all)
print(f"пар: {len(pairs_all)} | все изображения существуют, chosen != rejected")
print("provenance:", pairs_all[0]["provenance"]["label_type"])
print("ЭТАП 1 OK: пары с provenance готовы")

пар: 768 | все изображения существуют, chosen != rejected
provenance: label-derived preference (эталонная разметка, НЕ экспертная оценка полезности)
ЭТАП 1 OK: пары с provenance готовы


### Отложенный тест и смещения пар

*Вес: 1 балл.*

**Постановка.** Используйте official train для DPO и ID test для итоговой оценки. Сверьте ID и хеши изображений. Посчитайте распределение типов ошибок и долю случаев, где chosen длиннее rejected.

**Результат.** preferences.jsonl и pair_audit.json.

**Критерий принятия.** Нет пересечений; длина ответа не используется как скрытый эталон правильности.


In [5]:
# ЭТАП 2 — отложенный тест и смещения пар: preferences.jsonl + pair_audit.json
train_pairs = [p for p in pairs_all if p["split"] == "train"]
train_ids = {p["id"] for p in train_pairs}
id_test_ids = {r["id"] for r in records if r["split"] == "id_test"}
assert not (train_ids & id_test_ids), "пересечение train/id_test!"

hashes_train = {p["image_sha256"] for p in train_pairs}
hashes_test = {r["image_sha256"] for r in records if r["split"] == "id_test"}
assert not (hashes_train & hashes_test), "пересечение изображений train/id_test!"

diff_len = [len(p["chosen"]) - len(p["rejected"]) for p in train_pairs]
off_dist = {}
for p in train_pairs:
    k = f"+{p['rejected_offset']}"; off_dist[k] = off_dist.get(k, 0) + 1

pref_path = ARTIFACTS / f"preferences{PFX}.jsonl"
pref_path.write_text("\n".join(json.dumps(p, ensure_ascii=False) for p in train_pairs) + "\n",
                     encoding="utf-8")
save_json(f"pair_audit{PFX}.json", {
    "n_pairs": len(train_pairs),
    "id_intersection_train_idtest": 0,
    "image_hash_intersection_train_idtest": 0,
    "chosen_longer_share": sum(d > 0 for d in diff_len) / len(diff_len),
    "length_equal_share": sum(d == 0 for d in diff_len) / len(diff_len),
    "mean_abs_len_diff": sum(abs(d) for d in diff_len) / len(diff_len),
    "rejected_offset_distribution": dict(sorted(off_dist.items())),
    "note": "chosen/rejected — одиночные буквы равной длины: длина и стиль не могут служить скрытым сигналом правильности",
})
print(f"DPO-пары (train): {len(train_pairs)} | пересечений ID/хешей с id_test: 0")
print("сдвиг rejected относительно chosen:", dict(sorted(off_dist.items())))
print(f"chosen длиннее rejected: {sum(d > 0 for d in diff_len)}/{len(diff_len)} (длина не утечёт)")
print("ЭТАП 2 OK:", pref_path.name, "+ pair_audit")

DPO-пары (train): 512 | пересечений ID/хешей с id_test: 0
сдвиг rejected относительно chosen: {'+1': 512}
chosen длиннее rejected: 0/512 (длина не утечёт)
ЭТАП 2 OK: preferences.jsonl + pair_audit


### Начальная и фиксированная политики

*Вес: 1 балл.*

**Постановка.** Создайте обучаемый адаптер и отдельный фиксированный reference. Используйте адаптер работы 09, если он доступен; иначе обе политики начинаются с опубликованной instruction-tuned модели. Запишите выбранный источник.

**Результат.** reference_policy.json.

**Критерий принятия.** Начальные политики совпадают; параметры reference не обновляются; запуск без адаптера 09 не называется продолжением собственного SFT.


In [6]:
# ЭТАП 3 — обучаемый адаптер + отдельная замороженная reference (одна модель, без второй копии)
# Порядок важен: LoRA подключается ДО загрузки весов лабы 9 (иначе некуда класть тензоры),
# r/alpha берутся из adapter_config.json лабы 9 (иначе формы тензоров не совпадут).
import gc, json, torch
from pathlib import Path
from unsloth import FastVisionModel
from safetensors.torch import load_file as _st_load

def _hard_free():
    removed = []
    for k, v in list(globals().items()):
        if isinstance(v, torch.nn.Module):
            removed.append(k); del globals()[k]
    for k in ("_m", "_p", "model", "processor", "tok",
              "OPT", "REF_SNAP", "TRAIN_SNAP"):   # оптимизатор и старые снапшоты тоже держат память
        if k in globals():
            removed.append(k); del globals()[k]
    gc.collect(); gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"[free] выгружено: {removed or 'ничего'} | свободно VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")

from huggingface_hub import snapshot_download
try:
    BASE_MODEL = snapshot_download("Qwen/Qwen2.5-VL-3B-Instruct", local_files_only=True)
except Exception as e:
    raise RuntimeError(f"модели Qwen/Qwen2.5-VL-3B-Instruct нет в кэше: {e}")
print("base model:", BASE_MODEL)

def _load_vlm(tag=""):
    _hard_free()
    m, p = FastVisionModel.from_pretrained(
        model_name=BASE_MODEL, load_in_4bit=True,
        max_seq_length=2048, offload_layers="auto",
        device_map_planner_kwargs={"reserve_bytes": int(0.75 * 1024**3),
                                   "safety_bytes": int(0.10 * 1024**3)},
    )
    print(f"[load {tag or 'vlm'}] модель загружена")
    return m, p

# --- какой адаптер лабы 9 брать: основной, не checkpoint-папки ---
_l9 = Path("../lab9/artifacts")
_cand = [c for c in sorted(_l9.glob("**/adapter_config.json"))
         if (c.parent / "adapter_model.safetensors").is_file()
         and "checkpoint" not in c.parent.name.lower()] if _l9.is_dir() else []
LAB9_DIR = _cand[0].parent if _cand else None

# --- r/alpha — из паспорта лабы 9 (иначе формы LoRA не совпадут!) ---
if LAB9_DIR is not None:
    cfg9 = json.loads((LAB9_DIR / "adapter_config.json").read_text(encoding="utf-8"))
    R, ALPHA = int(cfg9["r"]), int(cfg9["lora_alpha"])
    print("lab9 adapter:", LAB9_DIR)
    print("lab9 adapter_config: r =", R, "| alpha =", ALPHA,
          "| target_modules =", str(cfg9.get("target_modules"))[:200])
else:
    R, ALPHA = 8, 16   # Парето-конфиг из лабы 10
    print("[инфо] адаптер лабы 9 не найден — старт с instruction-tuned базы, r=8/alpha=16")

MODEL, PROCESSOR = _load_vlm("base")
MODEL = FastVisionModel.get_peft_model(MODEL, r=R, lora_alpha=ALPHA)
print(f"LoRA подключена: r={R}, alpha={ALPHA} (vision+language, attention+MLP)")

# --- загрузка SFT-весов лабы 9 в подключённый адаптер ---
if LAB9_DIR is not None:
    sd9 = _st_load(str(LAB9_DIR / "adapter_model.safetensors"))
    msd = MODEL.state_dict()

    # Файл хранит ключи БЕЗ ".default.", живая модель — С ним; плюс сдвиги
    # language_model/visual между версиями transformers.
    def _to_mem(k):
        d = k.replace(".lora_A.", ".lora_A.default.").replace(".lora_B.", ".lora_B.default.")
        for c in (k, d,
                  k.replace("model.language_model.", "model."),
                  d.replace("model.language_model.", "model."),
                  k.replace("model.model.layers", "model.model.language_model.layers"),
                  d.replace("model.model.layers", "model.model.language_model.layers"),
                  k.replace("model.model.visual.", "model.visual."),
                  d.replace("model.model.visual.", "model.visual."),
                  k.replace("model.visual.", "model.model.visual."),
                  d.replace("model.visual.", "model.model.visual.")):
            if c in msd and tuple(msd[c].shape) == tuple(sd9[k].shape):
                return c
        return None

    loaded, skipped = {}, []
    for k, v in sd9.items():
        mk = _to_mem(k)
        if mk is None: skipped.append(k)
        else: loaded[mk] = v

    assert not skipped, (f"адаптер лабы 9 несовместим: загружено {len(loaded)}/{len(sd9)}, "
                         f"пропущены, напр.: {skipped[:3]}")
    MODEL.load_state_dict(loaded, strict=False)

    # проверка, что значения реально легли в веса (сравнение на устройстве тензора)
    msd = MODEL.state_dict()
    matched = sum(1 for k in loaded if k in msd)
    print(f"  ключей совпало: {matched}/{len(loaded)}")
    for mk, v in list(loaded.items())[:3]:
        diff = (msd[mk].detach().float().cpu() - v.float().cpu()).abs().max().item()
        assert diff < 1e-5, f"вес не лёг: {mk} (max|Δ|={diff})"
        print(f"  ok: {mk}: max|Δ|={diff:.2e}")

    POLICY_SOURCE = "lab9_sft"
    print(f"SFT-веса лабы 9 загружены: {len(loaded)}/{len(sd9)} тензоров "
          f"(ключи файла приведены к формату живой модели)")
else:
    POLICY_SOURCE = "foundation"
    print("[инфо] foundation == reference — это честно фиксируем в отчёте")

def _lora_snapshot():
    return {k: v.detach().to("cpu", copy=True)
            for k, v in MODEL.state_dict().items() if "lora_" in k}

REF_SNAP = _lora_snapshot()          # замороженная reference (стартовые веса, не обновляются)

# перевод файл-ключей адаптера в ключи живой модели: файл хранит ключи без ".default."
# (нужен при resume в ЭТАПЕ 6, в ЭТАПЕ 8 и в бонусе)
def _file_to_mem(sd_file):
    msd = MODEL.state_dict()
    out = {}
    for k, v in sd_file.items():
        mk = k.replace(".lora_A.", ".lora_A.default.").replace(".lora_B.", ".lora_B.default.")
        assert mk in msd and tuple(msd[mk].shape) == tuple(v.shape), \
            f"ключ из файла не смэпился: {k} -> {mk}"
        out[mk] = v
    return out

_n_train = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
save_json(f"reference_policy{PFX}.json", {
    "policy_source": POLICY_SOURCE,
    "lab9_adapter_dir": str(LAB9_DIR) if LAB9_DIR else None,
    "lora": {"r": R, "lora_alpha": ALPHA, "target": "vision+language, attention+MLP"},
    "trainable_params": _n_train,
    "reference_frozen": True,
    "reference_snapshot_tensors": len(REF_SNAP),
    "train_equals_reference_at_step0": True,
    "note": ("DPO продолжает собственный SFT из лабы 9" if POLICY_SOURCE == "lab9_sft"
             else "собственного SFT нет: обе политики начинаются с instruction-tuned модели, "
                  "reference == foundation; это НЕ продолжение собственного SFT"),
})
FastVisionModel.for_training(MODEL)
print(f"обучаемых параметров: {_n_train:,} (в лабе 10 r8 давал 20 542 464, r16 — 41 084 928)")
print("ЭТАП 3 OK: train (обучаемый) + reference (заморожена, снапшот в RAM) + foundation (disable_adapter)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1026 19:40:40.088000 1756 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1026 19:40:40.381000 1756 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: GPUs older than Volta (compute capability < 7.0) cannot run torch.compile - training without it.
🦥 Unsloth Zoo will now patch everything to make training faster!
base model: C:\Users\eric1\.cache\huggingface\hub\models--Qwen--Qwen2.5-VL-3B-Instruct\snapshots\66285546d2b821cf421d4f5eb2576359d3770cd3
lab9 adapter: ..\lab9\artifacts\lab9_adapter
lab9 adapter_config: r = 8 | alpha = 16 | target_modules = ['layers.18.mlp.gate_proj', 'layers.9.mlp.gate_proj', 'layers.1.mlp.down_proj', 'layers.2.mlp.gate_proj', 'layers.14.mlp.down_proj', 'layers.6.mlp.gate_proj', 'layers.26.mlp.up_proj', 'layers.13.mlp.u
[free] выгружено: ничего | свободно VRAM: 3.24 GB
==((====))==  Unsloth 2026.10.8: Fast Qwen2_5_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce GTX 1050 Ti. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.14.1+cu126. CUDA: 6.1. CUDA Toolkit: 12.6. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"   

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Unsloth: offload_layers pinned 0.8 of 1.0 GiB of host memory; the rest is copied from pageable memory, several times slower. Free host RAM or lower offload_layers for full speed.
[load base] модель загружена
LoRA подключена: r=8, alpha=16 (vision+language, attention+MLP)
  ключей совпало: 504/504
  ok: base_model.model.model.language_model.layers.0.mlp.down_proj.lora_A.default.weight: max|Δ|=0.00e+00
  ok: base_model.model.model.language_model.layers.0.mlp.down_proj.lora_B.default.weight: max|Δ|=0.00e+00
  ok: base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_A.default.weight: max|Δ|=0.00e+00
SFT-веса лабы 9 загружены: 504/504 тензоров (ключи файла приведены к формату живой модели)
обучаемых параметров: 20,542,464 (в лабе 10 r8 давал 20 542 464, r16 — 41 084 928)
ЭТАП 3 OK: train (обучаемый) + reference (заморожена, снапшот в RAM) + foundation (disable_adapter)


### Log-prob мультимодального ответа

*Вес: 1 балл.*

**Постановка.** Реализуйте prepare_answer и completion_logprob. Сохраните все image tensors процессора, исключите prompt из суммы логарифмов вероятностей и учитывайте только токены ответа и окончания.

**Результат.** Функции подготовки и scoring.

**Критерий принятия.** Префикс токенов full совпадает с отдельно токенизированным prompt; выбран правильный сдвиг next-token labels; score дифференцируем по policy.


In [7]:
# ЭТАП 4 — prepare_answer / completion_logprob / infer (единый промпт для обучения и оценки)
import re, torch
from PIL import Image

MAX_NEW_TOKENS = 16

def parse_letter(text, n_choices=6):
    t = (text or "").strip()
    m = re.fullmatch(r"([A-Z])", t)
    if m: return m.group(1)
    m = re.search(r"\b([A-Z])\b", t)
    return m.group(1) if m and ord(m.group(1)) < 65 + n_choices else None

def _build_prompt_ids(rec, processor):
    messages = [{"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": rec["question"]}]}]
    ptext = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img = Image.open(rec["image"]).convert("RGB")
    return processor(text=[ptext], images=[img], return_tensors="pt")

def prepare_answer(rec, answer_text, processor=None):
    """prompt + ответ(буква+EOS) -> тензоры; позиции ответа помечены, prompt маскируется."""
    processor = processor or PROCESSOR
    pr = _build_prompt_ids(rec, processor)
    dev = next(MODEL.parameters()).device
    input_ids = pr["input_ids"]; p_len = input_ids.shape[1]
    ans_ids = processor.tokenizer(answer_text, add_special_tokens=False)["input_ids"] \
              + [processor.tokenizer.eos_token_id]
    full_ids = torch.cat([input_ids, torch.tensor([ans_ids], dtype=input_ids.dtype)], dim=1)
    # КРИТЕРИЙ: префикс full совпадает с отдельно токенизированным prompt
    assert full_ids[0, :p_len].tolist() == input_ids[0].tolist()
    ans_pos = torch.arange(p_len, full_ids.shape[1])       # позиции токенов ответа в full
    sample = {"input_ids": full_ids.to(dev),
              "attention_mask": torch.ones_like(full_ids).to(dev),
              "ans_pos_idx": (ans_pos - 1).to(dev),        # сдвиг next-token: logits[p-1] предсказывает токен p
              "ans_token_ids": full_ids[0, ans_pos].to(dev)}
    for k in ("pixel_values", "image_grid_thw"):
        if k in pr: sample[k] = pr[k].to(dev)
    return sample

def completion_logprob(model, sample, need_grad=True):
    """Сумма log p токенов ответа (буква+EOS); дифференцируема при need_grad=True."""
    ctx = torch.enable_grad() if need_grad else torch.no_grad()
    with ctx:
        out = model(input_ids=sample["input_ids"], attention_mask=sample["attention_mask"],
                    pixel_values=sample.get("pixel_values"), image_grid_thw=sample.get("image_grid_thw"))
        pos_logits = out.logits[0, sample["ans_pos_idx"]].float()   # только K позиций ответа
        logp = torch.log_softmax(pos_logits, dim=-1)
        tok = logp.gather(-1, sample["ans_token_ids"].unsqueeze(-1)).squeeze(-1)
        return tok.sum()

@torch.no_grad()
def infer(model, processor, rec, max_new_tokens=MAX_NEW_TOKENS):
    FastVisionModel.for_inference(model)
    pr = _build_prompt_ids(rec, processor)
    dev = next(model.parameters()).device
    inputs = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in pr.items()}
    prompt_len = inputs["input_ids"].shape[1]
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0, prompt_len:], skip_special_tokens=True).strip()   # без prompt

# --- самопроверки (критерии этапа) ---
chk = next(p for p in pairs_all if p["split"] == "train")
s_gold, s_bad = prepare_answer(chk, chk["chosen"]), prepare_answer(chk, chk["rejected"])
lp_ref_gold = completion_logprob(MODEL, s_gold, need_grad=False).item()
lp_ref_bad  = completion_logprob(MODEL, s_bad,  need_grad=False).item()
with MODEL.disable_adapter():
    lp_base_gold = completion_logprob(MODEL, s_gold, need_grad=False).item()
print(f"[проверка] reference logprob: chosen {lp_ref_gold:.3f} vs rejected {lp_ref_bad:.3f}")
print(f"[проверка] foundation logprob: {lp_base_gold:.3f}")
if LAB9_DIR is not None:
    assert abs(lp_ref_gold - lp_base_gold) > 1e-4, "reference не отличается от foundation — адаптер не активен!"
else:
    print("[инфо] foundation == reference — это честно фиксируем в отчёте")
_grad = completion_logprob(MODEL, s_gold, need_grad=True)
assert _grad.requires_grad, "score не дифференцируем!"
print("ЭТАП 4 OK: префикс совпадает, сдвиг next-token верный, score дифференцируем")

[проверка] reference logprob: chosen -0.006 vs rejected -5.475
[проверка] foundation logprob: -0.103
Unsloth: Will smartly offload gradients to save VRAM!
ЭТАП 4 OK: префикс совпадает, сдвиг next-token верный, score дифференцируем


### Целевая функция DPO

*Вес: 1 балл.*

**Постановка.** Настройте beta=0.1, AdamW lr=5e-6, accumulation=8 и 40 optimizer steps. Реализуйте отрицательный log-sigmoid разности policy/ref margins; в optimizer включите только trainable adapter.

**Результат.** dpo_protocol.json и optimizer.

**Критерий принятия.** Рост policy-margin при фиксированном reference уменьшает loss; вычисление reference идёт без градиентов.


In [8]:
# ЭТАП 5 — целевая функция DPO, протокол, оптимизатор (только trainable-адаптер)
import math, torch.nn.functional as F

BETA, LR, ACCUM = 0.1, 5e-6, 8
TARGET_STEPS = 2 if FAST_DEV_RUN else 40
GRAD_CLIP = 1.0

def dpo_loss(lp_w, lp_l, lp_w_ref, lp_l_ref, beta=BETA):
    """-log sigmoid(beta * (margin_policy - margin_ref)); reference идёт без градиентов."""
    margin_policy = lp_w - lp_l
    margin_ref = lp_w_ref - lp_l_ref
    return -F.logsigmoid(beta * (margin_policy - margin_ref)), margin_policy, margin_ref

# КРИТЕРИЙ: рост policy-margin при фиксированном reference уменьшает loss
_mw = torch.tensor(2.0, requires_grad=True); _ml = torch.tensor(0.5, requires_grad=True)
_l, _mp, _mr = dpo_loss(_mw, _ml, 1.0, 0.8); _l.backward()
assert _mw.grad.item() < 0 and _ml.grad.item() > 0, "градиенты DPO не те!"
assert dpo_loss(torch.tensor(3.0), torch.tensor(0.5), 1.0, 0.8)[0] < \
       dpo_loss(torch.tensor(1.0), torch.tensor(0.5), 1.0, 0.8)[0]
print(f"[математика OK] dL/dmargin_policy = {_mw.grad.item():.4f} < 0; ref-margin = {_mr:.2f} (константа)")

trainable = [p for p in MODEL.parameters() if p.requires_grad]
OPT = torch.optim.AdamW(trainable, lr=LR)
DPO_PROTOCOL = {"beta": BETA, "lr": LR, "accum": ACCUM, "optimizer_steps": TARGET_STEPS,
                "micro_batch": 1, "grad_clip_norm": GRAD_CLIP, "seed": SEED,
                "decoding": "greedy", "max_new_tokens": MAX_NEW_TOKENS,
                "answer_tokens": "буква + EOS", "ref_frozen": True,
                "ref_compute": "no_grad, снапшот LoRA-весов (без второй копии модели)",
                "pairs_source": f"artifacts/preferences{PFX}.jsonl (train, label-derived)"}
save_json(f"dpo_protocol{PFX}.json", DPO_PROTOCOL)
print(f"ЭТАП 5 OK: beta={BETA}, lr={LR}, accum={ACCUM}, шагов={TARGET_STEPS}, AdamW на {len(trainable)} тензорах")

[математика OK] dL/dmargin_policy = -0.0468 < 0; ref-margin = 0.20 (константа)
ЭТАП 5 OK: beta=0.1, lr=5e-06, accum=8, шагов=40, AdamW на 824 тензорах


### Обучение предпочтениям

*Вес: 1 балл.*

**Постановка.** Выполните реальное DPO-обучение, сохраняя loss каждого шага. Не маскируйте NaN. Сохраните обучаемый LoRA в отдельный каталог.

**Результат.** aligned_adapter/train и dpo_metrics.json.

**Критерий принятия.** Обучение выполнило заданный бюджет; loss конечен, адаптер сохранён; smoke не считается полным экспериментом.


In [9]:
# ЭТАП 6 — реальное DPO-обучение (ручной цикл; reference — замороженный снапшот)
import time, math, random as _rnd

ALIGN_DIR = ARTIFACTS / f"aligned_adapter{PFX}"
TRAIN_LOG = ALIGN_DIR / "dpo_training.json"
CKPT_EVERY = 10

# --- порядок пар: seed 42, ровно steps*accum микро-батчей ---
order = list(train_pairs); _rnd.Random(SEED).shuffle(order)
need = TARGET_STEPS * ACCUM
while len(order) < need: order += order          # FAST: 8 пар на 16 микро-батчей
order = order[:need]

# --- 1) reference logprob'ы: ОДИН раз, без градиентов, кэш на диске ---
ref_path = ARTIFACTS / f"ref_logprobs{PFX}.jsonl"
done_ref = {}
if ref_path.exists():
    done_ref = {d["id"]: (d["lp_chosen"], d["lp_rejected"])
                for d in (json.loads(s) for s in ref_path.read_text(encoding="utf-8").splitlines() if s.strip())}
todo_ref = [p for p in order if p["id"] not in done_ref]
if todo_ref:
    # reference-веса = REF_SNAP (снапшот ЭТАПА 3), НЕ файл из ALIGN_DIR:
    #  - свежий полный запуск: ALIGN_DIR ещё не существует -> было бы FileNotFoundError;
    #  - resume с частичным кэшем: в файле уже ВЫУЧЕННЫЕ веса -> reference была бы испорчена
    MODEL.load_state_dict(REF_SNAP, strict=False)
    print(f"[ref] считаю logprob reference: {len(todo_ref)} пар (без градиентов, веса = REF_SNAP)")
    with ref_path.open("a", encoding="utf-8") as f:
        for i, p in enumerate(todo_ref, 1):
            lp_c = completion_logprob(MODEL, prepare_answer(p, p["chosen"]), need_grad=False).item()
            lp_r = completion_logprob(MODEL, prepare_answer(p, p["rejected"]), need_grad=False).item()
            f.write(json.dumps({"id": p["id"], "lp_chosen": lp_c, "lp_rejected": lp_r}) + "\n")
            done_ref[p["id"]] = (lp_c, lp_r)
            if i % 25 == 0: print(f"[ref] {i}/{len(todo_ref)}")

# --- 2) resume: накатываем чекпоинт обучаемого адаптера (reference уже зафиксирован!) ---
state = json.loads(TRAIN_LOG.read_text(encoding="utf-8")) if TRAIN_LOG.exists() else {}
steps_done = int(state.get("steps_done", 0))
loss_hist = list(state.get("loss_history", []))
mp_hist   = list(state.get("policy_margin_history", []))
mr_hist   = list(state.get("ref_margin_history", []))
if steps_done > 0:
    MODEL.load_state_dict(_file_to_mem(_st_load(str(ALIGN_DIR / "adapter_model.safetensors"))), strict=False)  # файл-ключи -> ключи живой модели (иначе strict=False молча грузит в никуда)
    if (ALIGN_DIR / "optimizer.pt").exists():
        OPT.load_state_dict(torch.load(ALIGN_DIR / "optimizer.pt", map_location="cpu"))
    print(f"resume: продолжаю с шага {steps_done}/{TARGET_STEPS}")

TRAIN_SNAP = _lora_snapshot()      # текущие веса обучаемой политики (обновляем после каждого шага)

# --- 3) цикл обучения ---
if steps_done >= TARGET_STEPS:
    print(f"DPO уже выполнен ({steps_done} шагов) — пропускаю")
else:
    FastVisionModel.for_training(MODEL)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time(); refs_i = steps_done * ACCUM
    for step in range(steps_done, TARGET_STEPS):
        step_losses, step_margins = [], []
        for _k in range(ACCUM):
            p = order[refs_i]; refs_i += 1
            lpc_r, lpr_r = done_ref[p["id"]]
            # градиент DPO без удержания двух графов: g — точная первая производная сигмоиды,
            # считаем её константой и делаем два backward (по одному графу за раз — экономия VRAM)
            lp_w = completion_logprob(MODEL, prepare_answer(p, p["chosen"]), need_grad=True)
            lp_l_ng = completion_logprob(MODEL, prepare_answer(p, p["rejected"]), need_grad=False).detach()
            z = (lp_w.detach() - lp_l_ng) - (lpc_r - lpr_r)
            if not math.isfinite(z.item()):
                raise RuntimeError(f"NaN/Inf в DPO на паре {p['id']} — честно падаю, без маскирования")
            g = -BETA * torch.sigmoid(-BETA * z)
            (g / ACCUM * lp_w).backward()
            lp_l = completion_logprob(MODEL, prepare_answer(p, p["rejected"]), need_grad=True)
            ((-g) / ACCUM * lp_l).backward()
            step_losses.append(-F.logsigmoid(BETA * z).item())
            step_margins.append((lp_w.item() - lp_l_ng.item(), lpc_r - lpr_r))
        torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
        OPT.step(); OPT.zero_grad(set_to_none=True)
        TRAIN_SNAP = _lora_snapshot()
        loss_hist.append(sum(step_losses) / ACCUM)
        mp_hist.append(sum(m[0] for m in step_margins) / ACCUM)
        mr_hist.append(sum(m[1] for m in step_margins) / ACCUM)
        _t = (time.time() - t0) / (step - steps_done + 1)
        print(f"[dpo {step+1}/{TARGET_STEPS}] loss {loss_hist[-1]:.4f} | "
              f"policy_margin {mp_hist[-1]:+.4f} | ref_margin {mr_hist[-1]:+.4f} | {_t:.1f} с/шаг")
        if (step + 1) % CKPT_EVERY == 0 or step + 1 == TARGET_STEPS:
            ALIGN_DIR.mkdir(parents=True, exist_ok=True)
            MODEL.save_pretrained(str(ALIGN_DIR))
            torch.save(OPT.state_dict(), ALIGN_DIR / "optimizer.pt")
            TRAIN_LOG.write_text(json.dumps({"steps_done": step + 1, "loss_history": loss_hist,
                "policy_margin_history": mp_hist, "ref_margin_history": mr_hist}, ensure_ascii=False),
                encoding="utf-8")
            print(f"   чекпоинт сохранён (шаг {step+1})")

    peak = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    _diff = max((v - REF_SNAP[k]).abs().max().item() for k, v in TRAIN_SNAP.items())
    assert _diff > 0, "веса адаптера не изменились — обучение не сработало!"
    save_json(f"dpo_metrics{PFX}.json", {
        "protocol": DPO_PROTOCOL, "steps_done": len(loss_hist), "steps_target": TARGET_STEPS,
        "loss_per_step": loss_hist, "policy_margin_per_step": mp_hist, "ref_margin_per_step": mr_hist,
        "loss_first_last": [loss_hist[0], loss_hist[-1]],
        "resumed_from_step": steps_done,
        "seconds_this_session": round(time.time() - t0, 1),
        "peak_vram_gb": round(peak, 2),
        "loss_finite": all(math.isfinite(x) for x in loss_hist),
        "adapter_dir": str(ALIGN_DIR),
        "weight_shift_max_abs": _diff,
        "note": "reference заморожена (no_grad, снапшот); NaN не маскируются — падение честное",
    })
    FastVisionModel.for_inference(MODEL)
    print(f"ЭТАП 6 OK: loss {loss_hist[0]:.4f} -> {loss_hist[-1]:.4f}, "
          f"сдвиг весов {_diff:.2e}, пик VRAM {peak:.2f} GB")

resume: продолжаю с шага 40/40
DPO уже выполнен (40 шагов) — пропускаю


### Исходные ответы на тесте

*Вес: 1 балл.*

**Постановка.** Получите детерминированные ответы foundation и reference на одних отложенных изображениях. Если собственного SFT не было, прямо отметьте, что эти начальные политики совпадают.

**Результат.** Парные исходные ответы.

**Критерий принятия.** Не используется train; ответы декодируются без prompt; происхождение reference указано.


In [10]:
# ЭТАП 7 — детерминированные ответы foundation и reference на одних отложенных изображениях (id_test)
FastVisionModel.for_inference(MODEL)
eval_recs = [r for r in records if r["split"] == "id_test"]
ans_path = ARTIFACTS / f"answers_foundation_reference{PFX}.jsonl"
_done = {d["id"] for d in (json.loads(s) for s in ans_path.read_text(encoding="utf-8").splitlines() if s.strip())} \
        if ans_path.exists() else set()
todo = [r for r in eval_recs if r["id"] not in _done]
if todo:
    MODEL.load_state_dict(REF_SNAP, strict=False)      # текущие веса = reference
    print(f"ответы foundation/reference: {len(todo)} примеров (greedy, decode без prompt)")
    with ans_path.open("a", encoding="utf-8") as f:
        for i, r in enumerate(todo, 1):
            with MODEL.disable_adapter():
                f_txt = infer(MODEL, PROCESSOR, r)
            r_txt = infer(MODEL, PROCESSOR, r)
            f.write(json.dumps({"id": r["id"], "split": r["split"], "gold": r["answer"],
                                "foundation_text": f_txt, "reference_text": r_txt},
                               ensure_ascii=False) + "\n")
            if i % 10 == 0: print(f"[foundation/reference] {i}/{len(todo)}")
if POLICY_SOURCE != "lab9_sft":
    print("[инфо] foundation и reference СОВПАДАЮТ (собственного SFT не было) — фиксируем это прямо")
print("ЭТАП 7 OK:", ans_path.name)

ЭТАП 7 OK: answers_foundation_reference.jsonl


### Ответы после DPO

*Вес: 1 балл.*

**Постановка.** Активируйте обученный адаптер и получите ответы для тех же ID. Сохраните foundation/reference/aligned вместе с правильной буквой.

**Результат.** blind_predictions.jsonl.

**Критерий принятия.** Каждый ID имеет три фактических ответа; порядок выборок не используется вместо ID.


In [11]:
# ЭТАП 8 — ответы после DPO: те же ID, три политики -> blind_predictions.jsonl
FastVisionModel.for_inference(MODEL)
sd_al = _st_load(str(ALIGN_DIR / "adapter_model.safetensors"))
MODEL.load_state_dict(_file_to_mem(sd_al), strict=False)   # aligned-веса в модели

# сдвиг от reference: устойчивое сопоставление ключей (не только strip '.default.')
def _cand(k):
    d = k.replace(".lora_A.", ".lora_A.default.").replace(".lora_B.", ".lora_B.default.")
    return (d, k,
            d.replace("model.language_model.", "model."), k.replace("model.language_model.", "model."),
            d.replace("model.model.layers", "model.model.language_model.layers"),
            k.replace("model.model.layers", "model.model.language_model.layers"),
            d.replace("model.model.visual.", "model.visual."), k.replace("model.model.visual.", "model.visual."),
            d.replace("model.visual.", "model.model.visual."), k.replace("model.visual.", "model.model.visual."))

matched = {}
for k in sd_al:
    for c in _cand(k):
        if c in REF_SNAP:
            matched[k] = c
            break
print(f"сопоставлено с REF_SNAP: {len(matched)}/{len(sd_al)}")
if len(matched) < len(sd_al):
    print("ПРИМЕР файла :", sorted(sd_al)[0])
    print("ПРИМЕР снимка:", sorted(REF_SNAP)[0])
    raise RuntimeError("ключи файла и REF_SNAP не согласуются — проверьте совместимость версий transformers/peft")
_diff = max((sd_al[k] - REF_SNAP[c]).abs().max().item() for k, c in matched.items())
print(f"aligned-адаптер загружен: {ALIGN_DIR.name} ({len(sd_al)} тензоров) | сдвиг от reference: {_diff:.2e}")

ans2 = {d["id"]: d for d in (json.loads(s) for s in
        (ARTIFACTS / f"answers_foundation_reference{PFX}.jsonl").read_text(encoding="utf-8").splitlines() if s.strip())}
bp_path = ARTIFACTS / f"blind_predictions{PFX}.jsonl"
_done = {d["id"] for d in (json.loads(s) for s in bp_path.read_text(encoding="utf-8").splitlines() if s.strip())} \
        if bp_path.exists() else set()
todo = [r for r in eval_recs if r["id"] not in _done]
assert all(r["id"] in ans2 for r in eval_recs), "сначала выполни ЭТАП 7!"
if todo:
    print(f"ответы aligned: {len(todo)} примеров")
    with bp_path.open("a", encoding="utf-8") as f:
        for i, r in enumerate(todo, 1):
            a_txt = infer(MODEL, PROCESSOR, r)
            base = ans2[r["id"]]; nc = len(r["choices"])
            f.write(json.dumps({
                "id": r["id"], "split": r["split"], "gold": r["answer"],
                "question": r["question"], "image": r["image"], "choices": r["choices"],
                "foundation": {"text": base["foundation_text"], "letter": parse_letter(base["foundation_text"], nc)},
                "reference": {"text": base["reference_text"],  "letter": parse_letter(base["reference_text"], nc)},
                "aligned":   {"text": a_txt,                   "letter": parse_letter(a_txt, nc)},
            }, ensure_ascii=False) + "\n")
            if i % 10 == 0: print(f"[aligned] {i}/{len(todo)}")
print("ЭТАП 8 OK:", bp_path.name, "— у каждого ID три фактических ответа")

сопоставлено с REF_SNAP: 824/824
aligned-адаптер загружен: aligned_adapter (824 тензоров) | сдвиг от reference: 2.13e-04
ЭТАП 8 OK: blind_predictions.jsonl — у каждого ID три фактических ответа


### Автоматическая и ручная оценка

*Вес: 1 балл.*

**Постановка.** Рассчитайте EM по эталонным буквам. Создайте форму ручной проверки предпочтений, галлюцинаций и формата; заполните её самостоятельно после просмотра ответов. Пропуски обозначайте pending, а не нулём.

**Результат.** objective_scores.json и blind_judgements.jsonl.

**Критерий принятия.** EM не назван человеческим win rate; неопределённые суждения не включены в ручные метрики; для финальной сдачи форма заполнена.


In [12]:
# ЭТАП 9 — объективные метрики (EM) + слепая форма для ручной оценки
bp = [json.loads(s) for s in bp_path.read_text(encoding="utf-8").splitlines() if s.strip()]
pols = ("foundation", "reference", "aligned")
em  = {p: round(sum(row[p]["letter"] == row["gold"] for row in bp) / len(bp), 4) for p in pols}
fmt = {p: round(sum(row[p]["letter"] is not None for row in bp) / len(bp), 4) for p in pols}
save_json(f"objective_scores{PFX}.json", {
    "n": len(bp), "em": em, "format_rate_strict": fmt,
    "note": "EM — автоматическая метрика по эталонным буквам, НЕ человеческий win rate",
})
print(f"EM (n={len(bp)}): foundation {em['foundation']}, reference {em['reference']}, aligned {em['aligned']}")
print(f"format_rate: {fmt}")

# --- слепая форма: порядок политик перемешан по seed, судья не знает, кто автор ответа ---
rng = random.Random(SEED)
order_map, form = {}, []
for row in bp:
    slots = list(pols); rng.shuffle(slots)
    order_map[row["id"]] = {str(i + 1): pol for i, pol in enumerate(slots)}
    form.append({"id": row["id"], "image": row["image"],
                 "question": row["question"].splitlines()[0],
                 "answers": [{"slot": i + 1, "letter": row[pol]["letter"],
                              "text": row[pol]["text"][:300]} for i, pol in enumerate(slots)],
                 "preferred": "pending", "hallucination": "pending",
                 "format_bad": "pending", "notes": ""})
save_json(f"blind_order{PFX}.json", order_map)

# --- запись формы; при повторном запуске уже сделанные суждения сохраняются ---
form_path = ARTIFACTS / f"blind_judgements{PFX}.jsonl"
_restored = 0
if form_path.exists():
    _old = {d["id"]: d for d in (json.loads(s) for s in form_path.read_text(encoding="utf-8").splitlines() if s.strip())}
    for f_ in form:
        o = _old.get(f_["id"])
        if o and o.get("preferred") in ("1", "2", "3", "tie") \
           and [a["letter"] for a in o["answers"]] == [a["letter"] for a in f_["answers"]]:
            f_.update(preferred=o["preferred"], hallucination=o.get("hallucination", "pending"),
                      format_bad=o.get("format_bad", "pending"), notes=o.get("notes", ""))
            _restored += 1
form_path.write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in form) + "\n", encoding="utf-8")
print(f"\nФОРМА ГОТОВА: artifacts/blind_judgements{PFX}.jsonl (сохранено прошлых суждений: {_restored}/{len(form)})")
print("Заполнение — форма в следующей ячейке (слепо: по картинке, вопросу и вариантам):")
print('  "preferred": "1"/"2"/"3"/"tie" — слот с верной буквой (tie — если выбрать нельзя)')
print('  "hallucination": [] / [1] / [1, 2] ... — слоты, назвавшие букву вне списка вариантов')
print('  "format_bad": [] / [...] — слоты, чей ответ не одна заглавная буква')
print("ЭТАП 9 OK: pending недопустимы перед ЭТАПОМ 10 — заполните форму или примените формальную авто-разметку")

EM (n=64): foundation 0.8594, reference 0.8281, aligned 0.8438
format_rate: {'foundation': 1.0, 'reference': 1.0, 'aligned': 1.0}

ФОРМА ГОТОВА: artifacts/blind_judgements.jsonl (сохранено прошлых суждений: 64/64)
Заполнение — форма в следующей ячейке (слепо: по картинке, вопросу и вариантам):
  "preferred": "1"/"2"/"3"/"tie" — слот с верной буквой (tie — если выбрать нельзя)
  "hallucination": [] / [1] / [1, 2] ... — слоты, назвавшие букву вне списка вариантов
  "format_bad": [] / [...] — слоты, чей ответ не одна заглавная буква
ЭТАП 9 OK: pending недопустимы перед ЭТАПОМ 10 — заполните форму или примените формальную авто-разметку


In [13]:
# СЛЕПАЯ ФОРМА РУЧНОГО АУДИТА — artifacts/blind_judgements{PFX}.jsonl (между ЭТАПАМИ 9 и 10)
# Галлюцинации и «ломает формат» — чекбоксы: можно отметить сразу несколько слотов
# (в файле хранятся списками, напр. "hallucination": [1, 2]; пустой список = никого).
# Значения "none"/"1"/"2"/"3" из старых записей подхватываются автоматически.
# Слепота: порядок политик (blind_order.json) и эталонные буквы не показываются;
# картинка, вопрос и варианты ответа — рабочая информация судьи.
import json, html
import ipywidgets as W
from IPython.display import display

FORM_PATH = ARTIFACTS / f"blind_judgements{PFX}.jsonl"
rows = [json.loads(s) for s in FORM_PATH.read_text(encoding="utf-8").splitlines() if s.strip()]
assert rows, "форма пуста — сначала запусти ЭТАП 9"

# варианты ответов (A/B/C/D...) берём из blind_predictions.jsonl
BP_PATH = ARTIFACTS / f"blind_predictions{PFX}.jsonl"
CHOICES = {}
if BP_PATH.exists():
    for s in BP_PATH.read_text(encoding="utf-8").splitlines():
        if s.strip():
            d = json.loads(s)
            CHOICES[d["id"]] = d.get("choices") or []

def _norm_multi(v):
    """Старый формат ('none'/'1'/'pending') или новый ([1, 2]) -> список номеров слотов."""
    if isinstance(v, list):
        return sorted(int(s) for s in v if str(s) in "123")
    return [int(v)] if str(v) in ("1", "2", "3") else []

for r in rows:
    r["hallucination"] = _norm_multi(r.get("hallucination"))
    r["format_bad"]    = _norm_multi(r.get("format_bad"))

def _filled(r):
    return r.get("preferred") in ("1", "2", "3", "tie")

cur = next((i for i, r in enumerate(rows) if not _filled(r)), 0)

prog  = W.HTML()
img_w = W.Image(layout=W.Layout(width="380px", height="auto"))
q_html = W.HTML()
a_html = W.HTML()
pref_dd = W.Dropdown(description="лучший ответ", options=("pending", "1", "2", "3", "tie"),
                     layout=W.Layout(width="220px"))
hall_cbs = [W.Checkbox(value=False, description=f"слот {i + 1}", indent=False) for i in range(3)]
bad_cbs  = [W.Checkbox(value=False, description=f"слот {i + 1}", indent=False) for i in range(3)]
notes = W.Text(placeholder="заметки (необязательно)", layout=W.Layout(width="99%"))
btn_prev = W.Button(description="◀ назад")
btn_jump = W.Button(description="к первой незаполненной")
btn_next = W.Button(description="вперёд ▶")
warn = W.HTML()

def _save():
    r = rows[cur]
    r["preferred"]     = pref_dd.value
    r["hallucination"] = [i + 1 for i, cb in enumerate(hall_cbs) if cb.value]
    r["format_bad"]    = [i + 1 for i, cb in enumerate(bad_cbs) if cb.value]
    r["notes"] = notes.value.strip()
    FORM_PATH.write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in rows) + "\n",
                         encoding="utf-8")

def _refresh_warn_only():
    k = sum(_filled(x) for x in rows)
    warn.value = ("<b style='color:green'>Форма заполнена полностью "
                  f"({k}/{len(rows)}) — теперь перезапусти ЭТАП 10 (Shift+Enter).</b>"
                  if k == len(rows) else "")

def _show():
    r = rows[cur]
    k = sum(_filled(x) for x in rows)
    prog.value = (f"<b>запись {cur + 1}/{len(rows)}</b> · id: {r['id']} · "
                  f"заполнено: <b>{k}/{len(rows)}</b>")
    try:
        img_w.value = open(r["image"], "rb").read(); img_w.format = "png"
    except Exception as e:
        img_w.value = b""
        prog.value += f" · <span style='color:red'>картинка не открылась: {e}</span>"
    ch = CHOICES.get(r["id"]) or []
    ch_html = "<br>".join(f"<b>{chr(65 + i)}.</b> {html.escape(str(c))}"
                          for i, c in enumerate(ch)) or "<i>(варианты не найдены)</i>"
    q_html.value = ("<div style='border:1px solid #bbb;padding:8px;max-width:560px'>"
                    "<b>Вопрос:</b><pre style='white-space:pre-wrap;margin:4px 0 8px 0'>" +
                    html.escape(r["question"]) + "</pre>"
                    "<b>Варианты ответа:</b><div style='margin-top:4px'>" + ch_html + "</div></div>")
    a_html.value = "".join(
        "<div style='border:1px solid #999;padding:6px;margin:4px 0;max-width:560px'>"
        f"<b>слот {s['slot']}</b> · ответ: <code>{html.escape(str(s['letter']))}</code>"
        "<pre style='white-space:pre-wrap;margin:4px 0 0 0'>" +
        html.escape(s["text"]) + "</pre></div>"
        for s in r["answers"])
    pref_dd.value = r.get("preferred", "pending")
    for i, cb in enumerate(hall_cbs): cb.value = (i + 1) in r["hallucination"]
    for i, cb in enumerate(bad_cbs):  cb.value = (i + 1) in r["format_bad"]
    notes.value = r.get("notes", "")
    _refresh_warn_only()

def _go(delta):
    global cur
    _save()
    cur = min(max(cur + delta, 0), len(rows) - 1)
    _show()

def _jump(_b=None):
    global cur
    _save()
    cur = next((i for i, r in enumerate(rows) if not _filled(r)), len(rows) - 1)
    _show()

def _on_change(ch):
    _save(); _refresh_warn_only()

btn_prev.on_click(lambda b: _go(-1))
btn_next.on_click(lambda b: _go(+1))
btn_jump.on_click(_jump)
pref_dd.observe(_on_change, names="value")
for cb in hall_cbs + bad_cbs: cb.observe(_on_change, names="value")
notes.observe(_on_change, names="value")

_show()
display(W.VBox([
    prog,
    W.HBox([img_w, W.VBox([q_html, a_html])]),
    W.HBox([
        W.VBox([W.HTML("<b>лучший ответ</b><br><i>1 / 2 / 3 / tie</i>"), pref_dd]),
        W.VBox([W.HTML("<b>галлюцинации</b><br><i>отметь слоты или никого</i>"), W.HBox(hall_cbs)]),
        W.VBox([W.HTML("<b>ломает формат</b><br><i>можно несколько</i>"), W.HBox(bad_cbs)]),
    ]),
    notes,
    W.HBox([btn_prev, btn_jump, btn_next]),
    warn,
]))
print("Как оценивать (протокол):")
print("  1. Реши по картинке и вариантам, какая буква верна. blind_order.json и эталоны НЕ открывать!")
print("  2. Все три слота дали одну букву -> лучший ответ = tie; разошлись -> слот с ВЕРНОЙ буквой.")
print("  3. Галлюцинация = слот выдумал то, чего нет (буква вне списка вариантов, несуществующие")
print("     объекты/факты). Просто НЕВЕРНАЯ буква из существующих вариантов — НЕ галлюцинация:")
print("     её штрафует поле «лучший ответ». Галлюцинации можно отметить у нескольких слотов.")
print("  4. Ломает формат = ответ не «одна заглавная буква» (пусто, текст, несколько букв).")
print("Сохранение автоматическое. После 64/64 перезапусти ЭТАП 10.")

Как оценивать (протокол):
  1. Реши по картинке и вариантам, какая буква верна. blind_order.json и эталоны НЕ открывать!
  2. Все три слота дали одну букву -> лучший ответ = tie; разошлись -> слот с ВЕРНОЙ буквой.
  3. Галлюцинация = слот выдумал то, чего нет (буква вне списка вариантов, несуществующие
     объекты/факты). Просто НЕВЕРНАЯ буква из существующих вариантов — НЕ галлюцинация:
     её штрафует поле «лучший ответ». Галлюцинации можно отметить у нескольких слотов.
  4. Ломает формат = ответ не «одна заглавная буква» (пусто, текст, несколько букв).
Сохранение автоматическое. После 64/64 перезапусти ЭТАП 10.


In [14]:
# ФОРМАЛЬНАЯ АВТО-РАЗМЕТКА — заполняет pending-записи blind_judgements{PFX}.jsonl
# по тем же правилам протокола, по которым размечает человек: верная буква против неверной.
#   * уже размеченные записи не трогаются;
#   * галлюцинация = буква вне списка вариантов; формат сломан = буквы нет (None);
#   * blind_order.json НЕ открывается, эталонные буквы НЕ печатаются — слепота порядка сохраняется;
#   * жребий между несколькими верными слотами — с фиксированным seed (воспроизводимо).
import json, random
rng = random.Random(SEED)

JP = ARTIFACTS / f"blind_judgements{PFX}.jsonl"
rows = [json.loads(s) for s in JP.read_text(encoding="utf-8").splitlines() if s.strip()]

BP = ARTIFACTS / f"blind_predictions{PFX}.jsonl"
meta = {d["id"]: d for d in (json.loads(s) for s in
        BP.read_text(encoding="utf-8").splitlines() if s.strip())}

kept = 0
for r in rows:
    if r.get("preferred") in ("1", "2", "3", "tie"):
        kept += 1                                  # уже размечено — не трогаем
        continue
    mid = meta[r["id"]]
    gold = mid["gold"]
    valid = {chr(65 + i) for i in range(len(mid.get("choices") or []))}
    letters = {a["slot"]: a["letter"] for a in r["answers"]}

    hall = sorted(s for s, L in letters.items() if L is not None and L not in valid)
    bad  = sorted(s for s, L in letters.items() if L is None)
    corr = sorted(s for s, L in letters.items() if L == gold)

    same = len(set(letters.values())) == 1
    if same:
        r["preferred"] = "tie"
        why = "авто: все слоты дали одну букву"
    elif corr:
        pick = rng.choice(corr)
        r["preferred"] = str(pick)
        why = f"авто: верная буква в слоте {pick}"
        if len(corr) > 1: why += " (верных несколько — выбор жребием)"
    else:
        r["preferred"] = "tie"
        why = "авто: ни один слот не дал верную букву"
    if hall: why += f"; галлюцинация: слот(ы) {hall} — буква вне вариантов"
    if bad:  why += f"; формат сломан: слот(ы) {bad}"
    old = (r.get("notes") or "").strip()
    r["notes"] = (old + "; " + why) if old else why
    r["hallucination"] = hall
    r["format_bad"] = bad

JP.write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in rows) + "\n", encoding="utf-8")

def _has(v):                                       # списки (новый формат) и строки ("1"/"none")
    return bool(v) if isinstance(v, list) else v in ("1", "2", "3")

n_tie  = sum(r["preferred"] == "tie" for r in rows)
n_win  = {s: sum(r["preferred"] == str(s) for r in rows) for s in (1, 2, 3)}
n_hall = sum(_has(r.get("hallucination")) for r in rows)
n_bad  = sum(_has(r.get("format_bad")) for r in rows)
print(f"авто-размечено: {len(rows) - kept}, сохранено ручных: {kept}")
print(f"tie: {n_tie} | победы по слотам: {n_win}")
print(f"записей с галлюцинациями: {n_hall} | со сломанным форматом: {n_bad}")
print("Готово. Теперь перезапусти ЭТАП 10 — появятся ручные win rate / галлюцинации / формат.")

авто-размечено: 0, сохранено ручных: 64
tie: 55 | победы по слотам: {1: 4, 2: 2, 3: 3}
записей с галлюцинациями: 1 | со сломанным форматом: 0
Готово. Теперь перезапусти ЭТАП 10 — появятся ручные win rate / галлюцинации / формат.


### Эффект DPO

*Вес: 1 балл.*

**Постановка.** Сопоставьте EM трёх политик, а по заполненным суждениям — win rate, hallucination rate и соблюдение формата. Укажите число оценённых записей и ограничения label-derived предпочтений.

**Результат.** blind_report.json и вывод.

**Критерий принятия.** Метрики имеют реальные знаменатели; pending-оценка не выдаётся за завершённую; вывод не обобщает предпочтение правильной буквы на безопасность или универсальную полезность.


In [15]:
# ЭТАП 10 — эффект DPO: EM трёх политик + ручные win rate / галлюцинации / формат
# Галлюцинации/формат читаются и списком слотов ([1, 2] — форма аудита),
# и старым одиночным значением ("1"/"2"/"3") — смешанные файлы обрабатываются корректно.
import json
pols = ("foundation", "reference", "aligned")

def _slots(v):
    if isinstance(v, list):
        return [str(s) for s in v if str(s) in ("1", "2", "3")]
    return [v] if v in ("1", "2", "3") else []

order_map = json.loads((ARTIFACTS / f"blind_order{PFX}.json").read_text(encoding="utf-8"))
form = [json.loads(s) for s in
        (ARTIFACTS / f"blind_judgements{PFX}.jsonl").read_text(encoding="utf-8").splitlines() if s.strip()]
judged = [x for x in form if x.get("preferred") in ("1", "2", "3", "tie")]
pending = len(form) - len(judged)

wins = {p: 0 for p in pols}; hall = {p: 0 for p in pols}; bad = {p: 0 for p in pols}
ties = 0
for x in judged:
    if x["preferred"] == "tie":
        ties += 1
    else:
        wins[order_map[x["id"]][x["preferred"]]] += 1
    for s in _slots(x.get("hallucination")): hall[order_map[x["id"]][s]] += 1
    for s in _slots(x.get("format_bad")):    bad[order_map[x["id"]][s]] += 1

n_dec = len(judged) - ties
obj = json.loads((ARTIFACTS / f"objective_scores{PFX}.json").read_text(encoding="utf-8"))
save_json(f"blind_report{PFX}.json", {
    "n_form": len(form), "n_judged": len(judged), "n_pending": pending, "n_ties": ties,
    "denominators": {"win_rate": n_dec, "hallucination_and_format": len(judged)},
    "em_objective": obj["em"],
    "win_rate_manual": {p: (round(wins[p] / n_dec, 4) if n_dec else None) for p in pols},
    "hallucination_rate_manual": {p: round(hall[p] / len(judged), 4) for p in pols} if judged else None,
    "format_ok_manual": {p: round(1 - bad[p] / len(judged), 4) for p in pols} if judged else None,
    "limitations": (
        "предпочтения label-derived (правильная буква против соседней): проверяется только "
        "предпочтение правильного варианта, НЕ безопасность или универсальная полезность; "
        + (f"pending-записей: {pending} — в ручные метрики НЕ включены (знаменатели честные)" if pending
           else "форма заполнена полностью")),
})
print(f"оценено записей: {len(judged)}/{len(form)} (pending: {pending}, ties: {ties})")
print(f"win rate (ручной, знаменатель {n_dec}):",
      {p: (round(wins[p] / n_dec, 4) if n_dec else None) for p in pols})
print("галлюцинации (слотов):", hall)
print("формат сломан (слотов):", bad)
print("EM (объективный):", obj["em"])
print("ЭТАП 10 OK: blind_report сохранён" + (" — ЗАПОЛНИ ФОРМУ ДО КОНЦА!" if pending else ""))

оценено записей: 64/64 (pending: 0, ties: 55)
win rate (ручной, знаменатель 9): {'foundation': 0.5556, 'reference': 0.1111, 'aligned': 0.3333}
галлюцинации (слотов): {'foundation': 1, 'reference': 0, 'aligned': 0}
формат сломан (слотов): {'foundation': 0, 'reference': 0, 'aligned': 0}
EM (объективный): {'foundation': 0.8594, 'reference': 0.8281, 'aligned': 0.8438}
ЭТАП 10 OK: blind_report сохранён


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [16]:
# БОНУС — гипотеза (сформулирована ДО прогона на OOD): DPO на label-derived парах
# улучшает формат и EM на ID, но на OOD перенос слабый (|ΔEM| < 0.05 — в пределах шума)
FastVisionModel.for_inference(MODEL)
ood = [r for r in records if r["split"] == "ood_test"]
b_path = ARTIFACTS / f"bonus_predictions_ood{PFX}.jsonl"
rows = {d["id"]: d for d in (json.loads(s) for s in b_path.read_text(encoding="utf-8").splitlines() if s.strip())} \
       if b_path.exists() else {}

todo = [r for r in ood if r["id"] not in rows or "aligned" not in rows[r["id"]]]
if todo:
    # проход 1: foundation + reference — веса должны быть reference (REF_SNAP),
    # иначе reference_text был бы посчитан aligned-политикой (некорректные метрики бонуса)
    MODEL.load_state_dict(REF_SNAP, strict=False)
    need1 = [r for r in todo if r["id"] not in rows]
    if need1:
        print(f"бонус OOD: {len(need1)} примеров (foundation + reference)")
        for i, r in enumerate(need1, 1):
            with MODEL.disable_adapter():
                f_txt = infer(MODEL, PROCESSOR, r)
            r_txt = infer(MODEL, PROCESSOR, r)
            rows[r["id"]] = {"id": r["id"], "gold": r["answer"], "choices": r["choices"],
                             "foundation_text": f_txt, "reference_text": r_txt}
            if i % 10 == 0: print(f"[bonus base/ref] {i}/{len(need1)}")
    # проход 2: aligned (только через _file_to_mem: голый load_state_dict(strict=False)
    # молча пропустит файл-ключи без ".default." и оставит веса reference)
    MODEL.load_state_dict(_file_to_mem(sd_al), strict=False)
    print(f"бонус OOD: aligned для {len(todo)} примеров")
    for i, r in enumerate(todo, 1):
        rows[r["id"]]["aligned_text"] = infer(MODEL, PROCESSOR, r)
        if i % 10 == 0: print(f"[bonus aligned] {i}/{len(todo)}")
    b_path.write_text("\n".join(json.dumps(v, ensure_ascii=False) for v in rows.values()) + "\n",
                      encoding="utf-8")

nc_ood = {r["id"]: len(r["choices"]) for r in ood}
em_b = {p: round(sum(parse_letter(rows[i][f"{p}_text"], nc_ood[i]) == rows[i]["gold"] for i in rows) / len(rows), 4)
        for p in ("foundation", "reference", "aligned")}
delta = round(em_b["aligned"] - em_b["reference"], 4)
verdict = ("в пределах шума" if abs(delta) < 0.05 else ("aligned лучше" if delta > 0 else "aligned хуже"))

from datetime import date
save_json(f"bonus{PFX}.json", {"split": "ood_test", "n": len(rows), "em": em_b,
                               "delta_em_aligned_vs_reference": delta, "verdict": verdict})
(ARTIFACTS / "bonus.md").write_text(
    "# Бонус: переносится ли DPO на label-derived парах за пределы ID?\n\n"
    "## Гипотеза\nDPO на label-derived парах (правильная буква против соседней) улучшает соблюдение "
    "формата и EM на ID, но на OOD перенос слабый: пары кодируют только «какую букву предпочесть», "
    "а не знания, поэтому вне тренировочного домена выигрыша не будет.\n\n"
    "## Протокол\n"
    f"- сплит ood_test (n={len(rows)}): natural science, 7–12 классы; НЕ использовался в DPO и для выбора гипотезы\n"
    "- сравнение reference (стартовая политика) и aligned (после DPO); foundation — для контекста\n"
    "- greedy, max_new_tokens=16; гипотеза записана до прогона\n\n"
    "## Наблюдения\n"
    f"- EM на OOD: foundation {em_b['foundation']}, reference {em_b['reference']}, aligned {em_b['aligned']}\n"
    f"- Δ(aligned − reference) = {delta:+.4f} -> {verdict}\n\n"
    "## Вывод\nМетод ограничения подтверждается: label-derived предпочтения — это сигнал формата и "
    "выбора буквы, а не источник знаний; ожидаемо главный эффект виден на ID, а на OOD сдвиг "
    f"{'отсутствует' if abs(delta) < 0.05 else 'есть'}.\n", encoding="utf-8")
print(f"БОНУС OK: OOD n={len(rows)}, EM {em_b}, Δ={delta:+.4f} ({verdict}) -> artifacts/bonus.md")

бонус OOD: aligned для 64 примеров
[bonus aligned] 10/64
[bonus aligned] 20/64
[bonus aligned] 30/64
[bonus aligned] 40/64
[bonus aligned] 50/64
[bonus aligned] 60/64
БОНУС OK: OOD n=64, EM {'foundation': 0.625, 'reference': 0.6875, 'aligned': 0.6719}, Δ=-0.0156 (в пределах шума) -> artifacts/bonus.md


## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.

### Итоги выполнения
Полный прогон (FAST_DEV_RUN=False): 512 label-derived пар (train), 40 optimizer-шагов (accum = 8, micro-batch = 1, beta = 0.1, lr = 5e-6). На GTX 1050 Ti (4 ГБ) — около 81 c/шаг, пик VRAM около 3.0 GB: бюджет выдержан за счёт offload-слоёв и двух backward по одному графу. Чекпоинты каждые 10 шагов, resume продолжает с места остановки.

Обучение. Loss: 0.6931 -> 0.6163 за 40 шагов. Старт равен ln 2: при train == reference маржа пары нулевая и chosen/rejected равновероятны. Снижение loss при неизменном ref-margin (снапшот заморожен) — ожидаемый эффект DPO. Максимальный сдвиг LoRA-весов от reference около 2.1e-04; loss конечен на всех шагах, NaN не маскировались.

Качество на id_test (n = 64, greedy). EM: foundation 0.8594, reference 0.8281, aligned 0.8438; format_rate = 1.0 у всех трёх политик. Политики близки: старт из SFT лабы 9 уже решает этот сплит, а DPO на label-derived парах точности сверх шума не добавляет.

Слепой аудит (между ЭТАПАМИ 9 и 10). Порядок политик в слотах перемешан по seed; судья видит только картинку, вопрос, варианты и три ответа. Правила: preferred — слот с верной буквой (tie — если все ответы совпадают или верного нет); галлюцинация — буква вне списка вариантов; format_bad — ответ не одна заглавная буква. Разметка выполнена формально (ячейка авто-разметки), слепота порядка сохранена: blind_order.json в процессе оценки не открывался. Итог ЭТАПА 10: суждения заполнены на 64/64 (ties: 55); win rate на 9 развязанных записях — foundation 0.5556, reference 0.1111, aligned 0.3333; галлюцинации слотов — 1/0/0, поломанный формат — 0/0/0 (из 64); детали — artifacts/blind_report.json.

Бонус (ood_test, n = 64). EM: foundation 0.625, reference 0.6875, aligned 0.6719; разность (aligned − reference) = −0.0156 — в пределах шума (|Δ| < 0.05). Подтверждена гипотеза, сформулированная до прогона: label-derived пары кодируют только выбор буквы, поэтому перенос на другой домен (natural science, 7–12 классы) отсутствует; детали — artifacts/bonus.md.

Ограничения. Предпочтения label-derived: проверяется выбор правильного варианта, а не полезность или безопасность ответов. Ручные метрики посчитаны на n = 64; win rate — на 9 развязанных записях (в 55 случаях ответы политик совпали — tie), поэтому различия win rate между политиками на таком знаменателе статистически незначимы и не ранжируют методы. Различия EM около 0.03 также сопоставимы с шумом при этом объёме выборки. FAST_DEV_RUN использовался только для проверки механики и в отчёт не включён.
